# Experiment 19 — Small-Lesion-Stratified Disease ΔAUC under Perturbation
**Purpose.** Test whether embedding-level artifact suppression preferentially
attenuates *small* clinical features. The motivating concern (microcalcifications
on mammograms, small SCLC nodules) predicts that signal loss is concentrated
on small lesions. EXP04 found near-zero global ΔAUC on lung_lesion, but did
not stratify by lesion size — leaving the small-feature hypothesis untested.
**Method.** Reuse cached clean linear probes and cached perturbed test
embeddings from EXP04. Use the NIH BBox subset (`Mass`+`Nodule` annotations,
which map to our `lung_lesion` label) to obtain ground-truth lesion area.
For each (model × perturbation):
  * Per-image score-shift: regress (clean − perturbed) probe score on
    log(lesion_area) across bbox-annotated positives. Negative slope ⇒
    smaller lesions suffer larger score drops.
  * Stratified ΔAUC: median-split bbox positives into small/large strata.
    Compute ΔAUC restricting positives to each stratum (negatives = full
    test). Gap = ΔAUC(small) − ΔAUC(large); positive ⇒ small lesions worse.
**Inputs needed (all already on disk after EXP04 finishes).**
  * `v4_exp04_clean_vs_perturbed_nih/disease_split.csv`
  * `v4_exp04_clean_vs_perturbed_nih/emb_clean_<model>_test.npy`
  * `v4_exp04_clean_vs_perturbed_nih/emb_pert_<model>_<pert>_test.npy`
  * `v4_exp04_clean_vs_perturbed_nih/disease_<model>_lung_lesion_clean_artefacts.pkl`
  * `/data0/NIH-CXR14/BBox_List_2017.csv`
**Output.** `v4_exp19_small_lesion_strata/` with three parquets:
`exp19_smalllesion_strata.parquet` (per-stratum ΔAUC),
`exp19_smalllesion_shifts.parquet` (per-image score shifts),
`exp19_smalllesion_corr.parquet` (Spearman ρ summary).
CPU-only; no GPU work. NIH-only by design (no bbox annotations on MIMIC).

In [ ]:
# === Papermill parameters (override via `papermill -p NAME VALUE`) ===
DATASET = "nih"            # one of: nih, mimic, emory
MODELS = "raddino,dinov2,dinov3,biomedclip,medsiglip"
SEED = 42
OUTPUTS_DIR = "/home/saptpurk/embeddings-noise-eliminators/outputs"
REPO_ROOT_OVERRIDE = "/home/saptpurk/embeddings-noise-eliminators"
HF_TOKEN_OVERRIDE = None     # set non-None only when running gated models locally


In [ ]:
# Apply papermill parameters to environment (no-op when env vars already set)
import os
os.environ.setdefault("DATASET", DATASET)
os.environ.setdefault("MODELS_TO_RUN", MODELS)
os.environ.setdefault("OUTPUTS_DIR", OUTPUTS_DIR)
os.environ.setdefault("REPO_ROOT", REPO_ROOT_OVERRIDE)
if HF_TOKEN_OVERRIDE:
    os.environ["HF_TOKEN"] = HF_TOKEN_OVERRIDE


In [1]:
import os
import pickle
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
from sklearn.metrics import roc_auc_score

ROOT = Path(os.environ.get(
    'V4_WORK_DIR', '/home/saptpurk/embeddings-noise-eliminators_work'))
DATASET = os.environ.get('DATASET', 'nih')
EXP04_DIR = ROOT / f'v4_exp04_clean_vs_perturbed_{DATASET}'
OUT = ROOT / 'v4_exp19_small_lesion_strata'
OUT.mkdir(parents=True, exist_ok=True)

LESION_FINDINGS = ('Nodule', 'Mass')
MODELS = ('raddino', 'dinov2', 'biomedclip', 'dinov3', 'medsiglip')
PERTURBATIONS = ('iso_blur_p4', 'iso_blur_p8',
                 'dir_blur_cranio_p32', 'dir_blur_cranio_p64',
                 'dir_blur_lateral_p32', 'dir_blur_lateral_p64',
                 'reticular_fine_p32', 'reticular_coarse_p64',
                 'ground_glass_p64')

## 1. Gate: NIH-only; bail with a manifest if inputs are absent

In [2]:
BBOX_CSV = Path('/data0/NIH-CXR14/BBox_List_2017.csv')
SPLIT_CSV = EXP04_DIR / 'disease_split.csv'

def _bail(status):
    pd.DataFrame({'status': [status]}).to_parquet(
        OUT / 'exp19_manifest.parquet', index=False)
    print(f'exp19 bailed: {status}')

if DATASET != 'nih':
    _bail('nih_only_by_design')
    raise SystemExit(0)
if not BBOX_CSV.exists() or not SPLIT_CSV.exists():
    _bail('inputs_absent')
    raise SystemExit(0)

## 2. Build bbox-annotated lung-lesion test subset with per-image area

In [3]:
bbox = pd.read_csv(BBOX_CSV, usecols=[0, 1, 2, 3, 4, 5])
bbox.columns = ['image', 'finding', 'x', 'y', 'w', 'h']
bbox['area_frac'] = (bbox['w'] * bbox['h']) / (1024.0 * 1024.0)

split = pd.read_csv(SPLIT_CSV)
split['image'] = split['image_path'].str.split('/').str[-1]
test_only = split[split['split'] == 'test'].reset_index(drop=True)
test_only['test_idx'] = np.arange(len(test_only))

bbox_test = bbox.merge(
    test_only[['image', 'test_idx', 'lung_lesion']], on='image', how='inner')
bbox_lesion = bbox_test[bbox_test['finding'].isin(LESION_FINDINGS)].copy()

# Per-image area: take the largest bbox if multiple exist for the same image.
bbox_per_img = bbox_lesion.groupby('image', as_index=False).agg(
    area_frac=('area_frac', 'max'),
    finding=('finding', lambda s: ','.join(sorted(set(s)))),
    test_idx=('test_idx', 'first'),
    lung_lesion=('lung_lesion', 'first'),
)
print(f'bbox-annotated lung-lesion test images: {len(bbox_per_img)}')
print(f'  area_frac quantiles: '
      f'{bbox_per_img["area_frac"].quantile([0,.25,.5,.75,1.0]).round(4).to_dict()}')

if len(bbox_per_img) < 10:
    _bail('too_few_bbox_test_cases')
    raise SystemExit(0)

med_area = bbox_per_img['area_frac'].median()
bbox_per_img['stratum'] = np.where(
    bbox_per_img['area_frac'] < med_area, 'small', 'large')
print(f'median-split at area_frac={med_area:.4f}: '
      f'{(bbox_per_img["stratum"]=="small").sum()} small, '
      f'{(bbox_per_img["stratum"]=="large").sum()} large')

y_test = test_only['lung_lesion'].values

bbox-annotated lung-lesion test images: 38
  area_frac quantiles: {0.0: 0.0011, 0.25: 0.0049, 0.5: 0.01, 0.75: 0.0177, 1.0: 0.2257}
median-split at area_frac=0.0100: 19 small, 19 large


## 3. Per-model, per-perturbation: stratified ΔAUC + per-image score shift

In [4]:
strata_records = []
shift_records = []
for model in MODELS:
    probe_pkl = EXP04_DIR / f'disease_{model}_lung_lesion_clean_artefacts.pkl'
    clean_emb = EXP04_DIR / f'emb_clean_{model}_test.npy'
    if not probe_pkl.exists() or not clean_emb.exists():
        warnings.warn(f'{model}: missing probe or clean emb; skipping')
        continue
    with open(probe_pkl, 'rb') as f:
        art = pickle.load(f)
    clf, scaler = art['classifier'], art['scaler']
    Xc = np.load(clean_emb)
    proba_c = clf.predict_proba(scaler.transform(Xc))[:, 1]
    auc_clean_full = roc_auc_score(y_test, proba_c)

    neg_idx = np.where(y_test == 0)[0]
    for pert in PERTURBATIONS:
        emb_p = EXP04_DIR / f'emb_pert_{model}_{pert}_test.npy'
        if not emb_p.exists():
            continue
        Xp = np.load(emb_p)
        proba_p = clf.predict_proba(scaler.transform(Xp))[:, 1]
        auc_pert_full = roc_auc_score(y_test, proba_p)

        for stratum in ('small', 'large'):
            pos = bbox_per_img[bbox_per_img['stratum'] == stratum]
            pos_idx = pos['test_idx'].values.astype(int)
            if len(pos_idx) == 0:
                continue
            keep = np.concatenate([pos_idx, neg_idx])
            y_sub = np.concatenate([np.ones(len(pos_idx)),
                                    np.zeros(len(neg_idx))])
            try:
                ac = roc_auc_score(y_sub, proba_c[keep])
                ap = roc_auc_score(y_sub, proba_p[keep])
            except ValueError:
                continue
            strata_records.append({
                'dataset': DATASET, 'model': model, 'perturbation': pert,
                'stratum': stratum, 'n_pos': int(len(pos_idx)),
                'auc_clean': ac, 'auc_perturbed': ap,
                'delta_auc': ac - ap,
                'auc_clean_fulltest': auc_clean_full,
                'auc_perturbed_fulltest': auc_pert_full,
                'delta_auc_fulltest': auc_clean_full - auc_pert_full,
            })

        for _, r in bbox_per_img.iterrows():
            i = int(r['test_idx'])
            shift_records.append({
                'dataset': DATASET, 'model': model, 'perturbation': pert,
                'image': r['image'], 'finding': r['finding'],
                'area_frac': float(r['area_frac']),
                'log_area': float(np.log(r['area_frac'])),
                'score_clean': float(proba_c[i]),
                'score_perturbed': float(proba_p[i]),
                'score_drop': float(proba_c[i] - proba_p[i]),
            })

strata_df = pd.DataFrame(strata_records)
shifts_df = pd.DataFrame(shift_records)

## 4. Spearman correlation: smaller lesions ⇒ larger score drop?

In [5]:
corr_records = []
if not shifts_df.empty:
    for (model, pert), sub in shifts_df.groupby(['model', 'perturbation']):
        if len(sub) < 6 or sub['score_drop'].std() == 0:
            continue
        rho, p = stats.spearmanr(sub['log_area'], sub['score_drop'])
        corr_records.append({
            'model': model, 'perturbation': pert,
            'n': int(len(sub)),
            'spearman_rho': float(rho), 'p_value': float(p),
        })
corr_df = pd.DataFrame(corr_records)

## 5. Save outputs and print headline summary

In [6]:
if strata_df.empty:
    _bail('no_strata_results')
    raise SystemExit(0)

strata_df.to_parquet(OUT / 'exp19_smalllesion_strata.parquet', index=False)
shifts_df.to_parquet(OUT / 'exp19_smalllesion_shifts.parquet', index=False)
if not corr_df.empty:
    corr_df.to_parquet(OUT / 'exp19_smalllesion_corr.parquet', index=False)

pivot = strata_df.pivot_table(
    index=['model', 'perturbation'], columns='stratum', values='delta_auc')
if 'small' in pivot.columns and 'large' in pivot.columns:
    pivot['gap_small_minus_large'] = pivot['small'] - pivot['large']
    summary = pivot.groupby('model')['gap_small_minus_large'].agg(
        ['mean', 'median', 'count']).round(4)
    summary.to_parquet(OUT / 'exp19_smalllesion_summary.parquet')
    print('\nMean ΔAUC gap (small − large) per model — positive ⇒ small lesions worse:')
    print(summary)

if not corr_df.empty:
    print('\nSpearman ρ(score_drop, log(area)) — negative ⇒ small lesions worse:')
    print(corr_df.pivot_table(
        index='model', columns='perturbation',
        values='spearman_rho').round(2))

print(f'\nwrote outputs under {OUT}')


Mean ΔAUC gap (small − large) per model — positive ⇒ small lesions worse:
              mean  median  count
model                            
biomedclip  0.0014  0.0016      9
dinov2      0.0000 -0.0004      9
dinov3     -0.0008  0.0003      9
medsiglip  -0.0027  0.0004      9
raddino    -0.0012 -0.0001      9

Spearman ρ(score_drop, log(area)) — negative ⇒ small lesions worse:
perturbation  dir_blur_cranio_p32  dir_blur_cranio_p64  dir_blur_lateral_p32  \
model                                                                          
biomedclip                  -0.15                -0.23                 -0.19   
dinov2                      -0.20                -0.21                 -0.31   
dinov3                      -0.14                -0.04                  0.06   
medsiglip                   -0.17                -0.16                 -0.23   
raddino                      0.09                 0.11                  0.10   

perturbation  dir_blur_lateral_p64  ground_glass_p64  iso